## generate dataset

generate_dataset.shの実行に時間がかかるのでその部分だけ切り出したもの。

### 事前準備

[Google Drive link](https://drive.google.com/drive/folders/1W5i5ryetj_gkcOpG1aZfL5Y8Yk6RxwYE)からデータをダウンロードし、下記のディレクトリ階層のように配置する。

<details>

```
/content/drive/MyDrive/OpenP5# tree -L 1
.
├── data
│   ├── Beauty
│   │   ├── item_collaborative_indexing_500_20_sequential.txt
│   │   ├── item_random_indexing.txt
│   │   ├── item_sequential_indexing_original.txt
│   │   ├── user_indexing.txt
│   │   ├── user_sequence_collaborative_indexing_500_20_sequential.txt
│   │   ├── user_sequence_random_indexing.txt
│   │   ├── user_sequence_sequential_indexing_original.txt
│   │   └── user_sequence.txt
│   ├── CDs
│   │   ├── item_collaborative_indexing_500_20_sequential.txt
│   │   ├── item_random_indexing.txt
│   │   ├── item_sequential_indexing_original.txt
│   │   ├── user_indexing.txt
│   │   ├── user_sequence_collaborative_indexing_500_20_sequential.txt
│   │   ├── user_sequence_random_indexing.txt
│   │   ├── user_sequence_sequential_indexing_original.txt
│   │   └── user_sequence.txt
│   ├── Clothing
│   │   ├── item_collaborative_indexing_500_20_sequential.txt
│   │   ├── item_random_indexing.txt
│   │   ├── item_sequential_indexing_original.txt
│   │   ├── user_indexing.txt
│   │   ├── user_sequence_collaborative_indexing_500_20_sequential.txt
│   │   ├── user_sequence_random_indexing.txt
│   │   ├── user_sequence_sequential_indexing_original.txt
│   │   └── user_sequence.txt
│   ├── Electronics
│   │   ├── item_collaborative_indexing_500_20_sequential.txt
│   │   ├── item_random_indexing.txt
│   │   ├── item_sequential_indexing_original.txt
│   │   ├── user_indexing.txt
│   │   ├── user_sequence_collaborative_indexing_500_20_sequential.txt
│   │   ├── user_sequence_random_indexing.txt
│   │   ├── user_sequence_sequential_indexing_original.txt
│   │   └── user_sequence.txt
│   ├── LastFM
│   │   ├── item_collaborative_indexing_500_20_sequential.txt
│   │   ├── item_random_indexing.txt
│   │   ├── item_sequential_indexing_original.txt
│   │   ├── user_indexing.txt
│   │   ├── user_sequence_collaborative_indexing_500_20_sequential.txt
│   │   ├── user_sequence_random_indexing.txt
│   │   ├── user_sequence_sequential_indexing_original.txt
│   │   └── user_sequence.txt
│   ├── ML100K
│   │   ├── item_collaborative_indexing_500_20_sequential.txt
│   │   ├── item_random_indexing.txt
│   │   ├── item_sequential_indexing_original.txt
│   │   ├── user_indexing.txt
│   │   ├── user_sequence_collaborative_indexing_500_20_sequential.txt
│   │   ├── user_sequence_random_indexing.txt
│   │   ├── user_sequence_sequential_indexing_original.txt
│   │   └── user_sequence.txt
│   ├── ML1M
│   │   ├── item_collaborative_indexing_500_20_sequential.txt
│   │   ├── item_random_indexing.txt
│   │   ├── item_sequential_indexing_original.txt
│   │   ├── user_indexing.txt
│   │   ├── user_sequence_collaborative_indexing_500_20_sequential.txt
│   │   ├── user_sequence_random_indexing.txt
│   │   ├── user_sequence_sequential_indexing_original.txt
│   │   └── user_sequence.txt
│   ├── Movies
│   │   ├── item_collaborative_indexing_500_20_sequential.txt
│   │   ├── item_random_indexing.txt
│   │   ├── item_sequential_indexing_original.txt
│   │   ├── user_indexing.txt
│   │   ├── user_sequence_collaborative_indexing_500_20_sequential.txt
│   │   ├── user_sequence_random_indexing.txt
│   │   ├── user_sequence_sequential_indexing_original.txt
│   │   └── user_sequence.txt
│   ├── Taobao
│   │   ├── item_collaborative_indexing_500_20_sequential.txt
│   │   ├── item_random_indexing.txt
│   │   ├── item_sequential_indexing_original.txt
│   │   ├── user_indexing.txt
│   │   ├── user_sequence_collaborative_indexing_500_20_sequential.txt
│   │   ├── user_sequence_random_indexing.txt
│   │   ├── user_sequence_sequential_indexing_original.txt
│   │   └── user_sequence.txt
│   └── Yelp
│       ├── item_collaborative_indexing_500_20_sequential.txt
│       ├── item_random_indexing.txt
│       ├── item_sequential_indexing_original.txt
│       ├── user_indexing.txt
│       ├── user_sequence_collaborative_indexing_500_20_sequential.txt
│       ├── user_sequence_random_indexing.txt
│       ├── user_sequence_sequential_indexing_original.txt
│       └── user_sequence.txt
└── preprocessed_data
```

</details>

このdriveをこのnotebookでマウントする

### 結果

`data`ディレクトリの対象データセットディレクトリに前処理済みのデータセットが保存される

In [ ]:
!git clone https://github.com/agiresearch/OpenP5.git

Cloning into 'OpenP5'...
remote: Enumerating objects: 478, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 478 (delta 79), reused 16 (delta 13), pack-reused 326
Receiving objects: 100% (478/478), 11.88 MiB | 11.05 MiB/s, done.
Resolving deltas: 100% (257/257), done.


In [ ]:
import argparse
import random
import argparse
import os
import torch
import torch.nn as nn
import torch.distributed as dist
from tqdm import tqdm
from collections import defaultdict
import logging
import re
import json
import pdb

import numpy as np
from itertools import combinations
from sklearn.cluster import SpectralClustering
from scipy.sparse import csr_matrix

import pickle
import inspect
import sys

In [ ]:
# from https://github.com/agiresearch/OpenP5/blob/main/src/utils/utils.py
def ReadLineFromFile(path):
    if not os.path.exists(path):
        raise FileNotFoundError
    lines = []
    with open(path,'r') as fd:
        for line in fd:
            lines.append(line.rstrip('\n'))
    return lines

def WriteDictToFile(path, write_dict):
    with open(path, 'w') as out:
        for user, items in write_dict.items():
            if type(items) == list:
                out.write(user + ' ' + ' '.join(items) + '\n')
            else:
                out.write(user + ' ' + str(items) + '\n')


def load_prompt_template(path, task_list):
    """
    Load prompt template from the file. Keep training tasks only.
    Input:
    - path: The path for prompt template txt file.
    - task_list: A list of required tasks.
    Return:
    - prompt_templates: a dictionary of prompt templates. e.g., {task: {'seen': {'0': {'Input': template_input, 'Output': template_output}}}}

    """

    if not os.path.exists(path):
        raise FileNotFoundError
    prompt_info = ReadLineFromFile(path)
    prompt_templates = dict()
    for prompt in prompt_info:
        t = [sens.strip() for sens in prompt.split(';')]
        if t[0] not in task_list:
            continue
        if t[0] not in prompt_templates:
            prompt_templates[t[0]] = dict()
        if t[1] not in prompt_templates[t[0]]:
            prompt_templates[t[0]][t[1]] = dict()
        num = len(prompt_templates[t[0]][t[1]])
        prompt_templates[t[0]][t[1]][str(num)] = dict()
        prompt_templates[t[0]][t[1]][str(num)]['Input'] = t[2]
        prompt_templates[t[0]][t[1]][str(num)]['Output'] = t[3]
    return prompt_templates

def get_info_from_prompt(prompt_templates):
    """
    Extract the require information from the prompt templates.
    Input:
    - prompt_templates: a dictionary of prompt templates.
    Output:
    - info: a list of required information.
    """

    info = []
    for task in prompt_templates:
        for see in prompt_templates[task]:
            for i in prompt_templates[task][see]:
                info += re.findall(r'\{.*?\}', prompt_templates[task][see][i]['Input'])
                info += re.findall(r'\{.*?\}', prompt_templates[task][see][i]['Output'])
    info = [i[1:-1] for i in set(info)]
    return info

def check_task_prompt(prompt_templates, task_list):
    """
    Check if all tasks have prompt templates. Raise Error if training tasks have no prompt.
    Input:
    - prompt_templates: A dictionary of prompt templates.
    - task_list: A list of training tasks.
    """
    for task in task_list:
        assert task in prompt_templates, f"No prompt for {task} task"

In [ ]:
# from https://github.com/agiresearch/OpenP5/blob/main/src/utils/indexing.py

def sequential_indexing(data_path, dataset, user_sequence_dict, order):
    """
    Use sequential indexing method to index the given user seuqnece dict.
    """
    user_index_file = os.path.join(data_path, dataset, 'user_indexing.txt')
    item_index_file = os.path.join(data_path, dataset, f'item_sequential_indexing_{order}.txt')
    reindex_sequence_file = os.path.join(data_path, dataset, f'user_sequence_sequential_indexing_{order}.txt')

    if os.path.exists(reindex_sequence_file):
        user_sequence = ReadLineFromFile(reindex_sequence_file)

        item_info = ReadLineFromFile(item_index_file)
        item_map = get_dict_from_lines(item_info)

        return construct_user_sequence_dict(user_sequence), item_map

    # For user index, load from txt file if already exists, otherwise generate from user sequence and save.
    if os.path.exists(user_index_file):
        user_info = ReadLineFromFile(user_index_file)
        user_map = get_dict_from_lines(user_info)
    else:
        user_map = generate_user_map(user_sequence_dict)
        WriteDictToFile(user_index_file, user_map)


    # For item index, load from txt file if already exists, otherwise generate from user sequence and save.
    if os.path.exists(item_index_file):
        item_info = ReadLineFromFile(item_index_file)
        item_map = get_dict_from_lines(item_info)
    else:
        item_map = dict()
        if order == 'original':
            user_list = user_sequence_dict.keys()
        elif order == 'short2long':
            user_list = sorted(user_sequence_dict, key=lambda x: len(user_sequence_dict[x]), reverse=False)
        elif order == 'long2short':
            user_list = sorted(user_sequence_dict, key=lambda x: len(user_sequence_dict[x]), reverse=True)

        for user in user_list:
            items = user_sequence_dict[user][:-2]
            for item in items:
                if item not in item_map:
                    item_map[item] = str(len(item_map) + 1001)
        for user in user_list:
            items = user_sequence_dict[user][-2:]
            for item in items:
                if item not in item_map:
                    item_map[item] = str(len(item_map) + 1001)
        WriteDictToFile(item_index_file, item_map)

    reindex_user_sequence_dict = reindex(user_sequence_dict, user_map, item_map)
    WriteDictToFile(reindex_sequence_file, reindex_user_sequence_dict)
    return reindex_user_sequence_dict, item_map



def random_indexing(data_path, dataset, user_sequence_dict):
    """
    Use random indexing method to index the given user seuqnece dict.
    """
    user_index_file = os.path.join(data_path, dataset, 'user_indexing.txt')
    item_index_file = os.path.join(data_path, dataset, 'item_random_indexing.txt')
    reindex_sequence_file = os.path.join(data_path, dataset, f'user_sequence_random_indexing.txt')

    if os.path.exists(reindex_sequence_file):
        user_sequence = ReadLineFromFile(reindex_sequence_file)

        item_info = ReadLineFromFile(item_index_file)
        item_map = get_dict_from_lines(item_info)

        return construct_user_sequence_dict(user_sequence), item_map

    # For user index, load from txt file if already exists, otherwise generate from user sequence and save.
    if os.path.exists(user_index_file):
        user_info = ReadLineFromFile(user_index_file)
        user_map = get_dict_from_lines(user_info)
    else:
        user_map = generate_user_map(user_sequence_dict)
        WriteDictToFile(user_index_file, user_map)


    # For item index, load from txt file if already exists, otherwise generate from user sequence and save.
    if os.path.exists(item_index_file):
        item_info = ReadLineFromFile(item_index_file)
        item_map = get_dict_from_lines(item_info)
    else:
        item_map = dict()
        items = set()
        for user in user_sequence_dict:
            items.update(user_sequence_dict[user])
        items = list(items)
        random.shuffle(items)
        for item in items:
            if item not in item_map:
                item_map[item] = str(len(item_map) + 1001)
        WriteDictToFile(item_index_file, item_map)

    reindex_user_sequence_dict = reindex(user_sequence_dict, user_map, item_map)
    WriteDictToFile(reindex_sequence_file, reindex_user_sequence_dict)
    return reindex_user_sequence_dict, item_map

def collaborative_indexing(data_path, dataset, user_sequence_dict, token_size, cluster_num, last_token, float32):
    """
    Use collaborative indexing method to index the given user seuqnece dict.
    """
    user_index_file = os.path.join(data_path, dataset, 'user_indexing.txt')
    item_index_file = os.path.join(data_path, dataset, f'item_collaborative_indexing_{token_size}_{cluster_num}_{last_token}.txt')
    reindex_sequence_file = os.path.join(data_path, dataset, f'user_sequence_collaborative_indexing_{token_size}_{cluster_num}_{last_token}.txt')

    if os.path.exists(reindex_sequence_file):
        user_sequence = ReadLineFromFile(reindex_sequence_file)

        item_info = ReadLineFromFile(item_index_file)
        item_map = get_dict_from_lines(item_info)

        return construct_user_sequence_dict(user_sequence), item_map

    # For user index, load from txt file if already exists, otherwise generate from user sequence and save.
    if os.path.exists(user_index_file):
        user_info = ReadLineFromFile(user_index_file)
        user_map = get_dict_from_lines(user_info)
    else:
        user_map = generate_user_map(user_sequence_dict)
        WriteDictToFile(user_index_file, user_map)


    # For item index, load from txt file if already exists, otherwise generate from user sequence and save.
    if os.path.exists(item_index_file):
        item_info = ReadLineFromFile(item_index_file)
        item_map = get_dict_from_lines(item_info)
    else:
        item_map = generate_collaborative_id(user_sequence_dict, token_size, cluster_num, last_token, float32)
        WriteDictToFile(item_index_file, item_map)

    reindex_user_sequence_dict = reindex(user_sequence_dict, user_map, item_map)
    WriteDictToFile(reindex_sequence_file, reindex_user_sequence_dict)
    return reindex_user_sequence_dict, item_map

def generate_collaborative_id(user_sequence_dict, token_size, cluster_num, last_token, float32):
    """
    Generate collaborative index for items.
    """
    # get the items in training data and all data.
    all_items = set()
    train_items = set()
    for user in user_sequence_dict:
        all_items.update(set(user_sequence_dict[user]))
        train_items.update(set(user_sequence_dict[user][:-2]))

    # reindex all training items for calculating the adjacency matrix
    item2id = dict()
    id2item = dict()
    for item in train_items:
        item2id[item] = len(item2id)
        id2item[len(id2item)] = item


    # calculate the co-occurrence of items in the training data as an adjacency matrix
    if float32 > 0:
        adj_matrix = np.zeros((len(item2id), len(item2id)), dtype=np.float32)
    else:
        adj_matrix = np.zeros((len(item2id), len(item2id)))
    for user in user_sequence_dict:
        interactions = user_sequence_dict[user][:-2]
        for pairs in combinations(interactions, 2):
            adj_matrix[item2id[pairs[0]]][item2id[pairs[1]]] += 1
            adj_matrix[item2id[pairs[1]]][item2id[pairs[0]]] += 1


    # get the clustering results for the first layer
    clustering = SpectralClustering(
        n_clusters=cluster_num,
        assign_labels="cluster_qr",
        random_state=0,
        affinity="precomputed",
    ).fit(adj_matrix)
    labels = clustering.labels_.tolist()

    # count the clustering results
    grouping = defaultdict(list)
    for i in range(len(labels)):
        grouping[labels[i]].append((id2item[i],i))

    item_map = dict()
    index_now = 0

    # add current clustering information into the item indexing results.
    item_map, index_now = add_token_to_indexing(item_map, grouping, index_now, token_size)

    # add current clustering info into a queue for BFS
    queue = []
    for group in grouping:
        queue.append(grouping[group])

    # apply BFS to further use spectral clustering for large groups (> token_size)
    while queue:
        group_items = queue.pop(0)

        # if current group is small enough, add the last token to item indexing
        if len(group_items) <= token_size:
            item_list = [items[0] for items in group_items]
            if last_token == 'sequential':
                item_map = add_last_token_to_indexing_sequential(item_map, item_list, token_size)
            elif last_token == 'random':
                item_map = add_last_token_to_indexing_random(item_map, item_list, token_size)
        else:
            # calculate the adjacency matrix for current group
            if float32 > 0:
                sub_adj_matrix = np.zeros((len(group_items), len(group_items)), dtype=np.float32)
            else:
                sub_adj_matrix = np.zeros((len(group_items), len(group_items)))
            for i in range(len(group_items)):
                for j in range(i+1, len(group_items)):
                    sub_adj_matrix[i][j] = adj_matrix[group_items[i][1]][group_items[j][1]]
                    sub_adj_matrix[j][i] = adj_matrix[group_items[j][1]][group_items[i][1]]

            # get the clustering results for current group
            clustering = SpectralClustering(
                n_clusters=cluster_num,
                assign_labels="cluster_qr",
                random_state=0,
                affinity="precomputed",
            ).fit(sub_adj_matrix)
            labels = clustering.labels_.tolist()

            # count current clustering results
            grouping = defaultdict(list)
            for i in range(len(labels)):
                grouping[labels[i]].append(group_items[i])

            # add current clustering information into the item indexing results.
            item_map, index_now = add_token_to_indexing(item_map, grouping, index_now, token_size)

            # push current clustering info into the queue
            for group in grouping:
                queue.append(grouping[group])

    # if some items are not in the training data, assign an index for them
    remaining_items = list(all_items - train_items)
    if len(remaining_items) > 0:
        if last_token == 'sequential':
            item_map = add_last_token_to_indexing_sequential(item_map, remaining_items, token_size)
        elif last_token == 'random':
            item_map = add_last_token_to_indexing_random(item_map, remaining_items, token_size)

    return item_map



def add_token_to_indexing(item_map, grouping, index_now, token_size):
    for group in grouping:
        index_now = index_now % token_size
        for (item, idx) in grouping[group]:
            if item not in item_map:
                item_map[item] = ''
            item_map[item] += f'<CI{index_now}>'
        index_now += 1
    return item_map, index_now

def add_last_token_to_indexing_random(item_map, item_list, token_size):
    last_tokens = random.sample([i for i in range(token_size)], len(item_list))
    for i in range(len(item_list)):
        item = item_list[i]
        if item not in item_map:
            item_map[item] = ''
        item_map[item] += f'<CI{last_tokens[i]}>'
    return item_map

def add_last_token_to_indexing_sequential(item_map, item_list, token_size):
    for i in range(len(item_list)):
        item = item_list[i]
        if item not in item_map:
            item_map[item] = ''
        item_map[item] += f'<CI{i}>'
    return item_map


def get_dict_from_lines(lines):
    """
    Used to get user or item map from lines loaded from txt file.
    """
    index_map = dict()
    for line in lines:
        info = line.split(" ")
        index_map[info[0]] = info[1]
    return index_map




def generate_user_map(user_sequence_dict):
    """
    generate user map based on user sequence dict.
    """
    user_map = dict()
    for user in user_sequence_dict.keys():
        user_map[user] = str(len(user_map) + 1)
    return user_map


def reindex(user_sequence_dict, user_map, item_map):
    """
    reindex the given user sequence dict by given user map and item map
    """
    reindex_user_sequence_dict = dict()
    for user in user_sequence_dict:
        uid = user_map[user]
        items = user_sequence_dict[user]
        reindex_user_sequence_dict[uid] = [item_map[i] for i in items]

    return reindex_user_sequence_dict


def construct_user_sequence_dict(user_sequence):
    """
    Convert a list of string to a user sequence dict. user as key, item list as value.
    """

    user_seq_dict = dict()
    for line in user_sequence:
        user_seq = line.split(" ")
        user_seq_dict[user_seq[0]] = user_seq[1:]
    return user_seq_dict

In [ ]:
# from https://github.com/agiresearch/OpenP5/blob/main/src/generate_dataset.py
def generate_dataset(
    data_path: str = '../data',
    item_indexing: str = "sequential",
    tasks_: str = 'sequential,straightforward',
    dataset: str = "Beauty",
    prompt_file: str = "../prompt.txt",
    sequential_order: str = "original",
    collaborative_token_size: int = 200,
    collaborative_cluster: int = 20,
    collaborative_last_token: str = "sequential",
    collaborative_float32: int = 0,
    max_his: int = 10,
    his_prefix: int = 1,
    his_sep:str = " , ",
    skip_empty_his: str = 1,
):
    tasks = tasks_.split(',')

    file_data = dict()
    file_data['data'] = []

    user_sequence = ReadLineFromFile(os.path.join(data_path, dataset, 'user_sequence.txt'))
    user_sequence_dict = construct_user_sequence_dict(user_sequence)

    if item_indexing == 'sequential':
        print("Reindex data with sequential indexing method")
        reindex_user_seq_dict, item_map = sequential_indexing(data_path, dataset, user_sequence_dict, sequential_order)
    elif item_indexing == 'random':
        print("Reindex data with random indexing method")
        reindex_user_seq_dict, item_map = random_indexing(data_path, dataset, user_sequence_dict)
    elif item_indexing == 'collaborative':
        print(f"Reindex data with collaborative indexing method with token_size {collaborative_token_size} and {collaborative_cluster} cluster")
        reindex_user_seq_dict, item_map = collaborative_indexing(data_path, dataset, user_sequence_dict, \
                                                                                    collaborative_token_size, collaborative_cluster, \
                                                                                    collaborative_last_token, collaborative_float32)
    else:
        raise NotImplementedError


    # get prompt
    prompt = load_prompt_template(prompt_file, tasks)
    info = get_info_from_prompt(prompt)
    check_task_prompt(prompt, tasks)
    print(f"get prompt from {prompt_file}")


    # Load training data samples
    training_data_samples = []
    for user in reindex_user_seq_dict:
        items = reindex_user_seq_dict[user][:-2]
        for i in range(len(items)):
            if i == 0:
                if skip_empty_his > 0:
                    continue
            one_sample = dict()
            one_sample['dataset'] = dataset
            one_sample['user_id'] = user
            if his_prefix > 0:
                one_sample['target'] = 'item_' + items[i]
            else:
                one_sample['target'] = items[i]
            if 'history' in info:
                history = items[:i]
                if max_his > 0:
                    history = history[-max_his:]
                if his_prefix > 0:
                    one_sample['history'] = his_sep.join(["item_" + item_idx for item_idx in history])
                else:
                    one_sample['history'] = his_sep.join(history)
            training_data_samples.append(one_sample)
    print("load training data")
    print(f'there are {len(training_data_samples)} samples in training data.')

    # construct sentences
    for i in range(len(training_data_samples)):
        one_sample = training_data_samples[i]
        for task in tasks:
            datapoint = {}
            datapoint['task'] = dataset + task
            datapoint['data_id'] = i
            for pid in prompt[task]['seen']:
                datapoint['instruction'] = prompt[task]['seen'][pid]['Input']
                datapoint['input'] = prompt[task]['seen'][pid]['Input'].format(**one_sample)
                datapoint['output'] = prompt[task]['seen'][pid]['Output'].format(**one_sample)
                file_data['data'].append(datapoint.copy())

    print("data constructed")
    print(f"there are {len(file_data['data'])} prompts in training data.")


    # save the data to json file
    output_path = f'{dataset}_{tasks_}_{item_indexing}_train.json'

    with open(os.path.join(data_path, dataset, output_path), 'w') as openfile:
        json.dump(file_data, openfile)


In [ ]:
# from https://github.com/agiresearch/OpenP5/blob/main/src/generate_dataset_eval.py

def generate_dataset_eval(
    data_path: str = '../data',
    item_indexing: str = "sequential",
    tasks_: str = 'sequential,straightforward',
    dataset: str = "Beauty",
    prompt_file: str = "../prompt.txt",
    sequential_order: str = "original",
    collaborative_token_size: int = 200,
    collaborative_cluster: int = 20,
    collaborative_last_token: str = "sequential",
    collaborative_float32: int = 0,
    max_his: int = 10,
    his_prefix: int = 1,
    his_sep:str = " , ",
    skip_empty_his: str = 1,
    mode: str = 'validation',
    prompt_str: str = 'seen:0',
):
    tasks = tasks_.split(',')

    file_data = dict()
    file_data['data'] = []

    user_sequence = ReadLineFromFile(os.path.join(data_path, dataset, 'user_sequence.txt'))
    user_sequence_dict = construct_user_sequence_dict(user_sequence)

    if item_indexing == 'sequential':
        print("Reindex data with sequential indexing method")
        reindex_user_seq_dict, item_map = sequential_indexing(data_path, dataset, user_sequence_dict, sequential_order)
    elif item_indexing == 'random':
        print("Reindex data with random indexing method")
        reindex_user_seq_dict, item_map = random_indexing(data_path, dataset, user_sequence_dict)
    elif item_indexing == 'collaborative':
        print(f"Reindex data with collaborative indexing method with token_size {collaborative_token_size} and {collaborative_cluster} cluster")
        reindex_user_seq_dict, item_map = collaborative_indexing(data_path, dataset, user_sequence_dict, \
                                                                                    collaborative_token_size, collaborative_cluster, \
                                                                                    collaborative_last_token, collaborative_float32)
    else:
        raise NotImplementedError


    # get prompt
    prompt = load_prompt_template(prompt_file, tasks)
    info = get_info_from_prompt(prompt)
    check_task_prompt(prompt, tasks)
    print(f"get prompt from {prompt_file}")

    # Load data samples
    if mode == 'validation':
        data_samples = load_validation(
            dataset=dataset,
            max_his=max_his,
            his_prefix=his_prefix,
            his_sep=his_sep,
            reindex_user_seq_dict=reindex_user_seq_dict,
            info=info
        )
        prompt_info = prompt_str.split(':')
        output_path = f'{dataset}_{tasks_}_{item_indexing}_validation_{prompt_str}.json'
    elif mode == 'test':
        data_samples = load_test(
            dataset,
            max_his,
            his_prefix,
            his_sep,
            reindex_user_seq_dict,
            info
        )
        prompt_info = prompt_str.split(':')
        output_path = f'{dataset}_{tasks_}_{item_indexing}_test_{prompt_str}.json'
    else:
        raise NotImplementedError
    print(f'there are {len(data_samples)} samples in {mode} data.')
    print(prompt_info)

    # construct sentences
    for i in range(len(data_samples)):
        one_sample = data_samples[i]
        for task in tasks:
            datapoint = {}
            datapoint['task'] = dataset + task
            datapoint['instruction'] = prompt[task][prompt_info[0]][prompt_info[1]]['Input']
            datapoint['input'] = prompt[task][prompt_info[0]][prompt_info[1]]['Input'].format(**one_sample)
            datapoint['output'] = prompt[task][prompt_info[0]][prompt_info[1]]['Output'].format(**one_sample)
            file_data['data'].append(datapoint.copy())

    print("data constructed")
    print(f"there are {len(file_data['data'])} prompts in {mode} data.")


    # save the data to json file

    with open(os.path.join(data_path, dataset, output_path), 'w') as openfile:
        json.dump(file_data, openfile)


def load_test(
    dataset,
    max_his,
    his_prefix,
    his_sep,
    reindex_user_seq_dict,
    info
    ):

    data_samples = []
    for user in reindex_user_seq_dict:
        items = reindex_user_seq_dict[user]
        one_sample = dict()
        one_sample['dataset'] = dataset
        one_sample['user_id'] = user
        if his_prefix > 0:
            one_sample['target'] = 'item_' + items[-1]
        else:
            one_sample['target'] = items[-1]
        if 'history' in info:
            history = items[:-1]
            if max_his > 0:
                history = history[-max_his:]
            if his_prefix > 0:
                one_sample['history'] = his_sep.join(["item_" + item_idx for item_idx in history])
            else:
                one_sample['history'] = his_sep.join(history)
        data_samples.append(one_sample)
    return data_samples

def load_validation(
    dataset,
    max_his,
    his_prefix,
    his_sep,
    reindex_user_seq_dict,
    info,
    ):
    data_samples = []
    for user in reindex_user_seq_dict:
        items = reindex_user_seq_dict[user]
        one_sample = dict()
        one_sample['dataset'] = dataset
        one_sample['user_id'] = user
        if his_prefix > 0:
            one_sample['target'] = 'item_' + items[-2]
        else:
            one_sample['target'] = items[-2]
        if 'history' in info:
            history = items[:-2]
            if max_his > 0:
                history = history[-max_his:]
            if his_prefix > 0:
                one_sample['history'] = his_sep.join(["item_" + item_idx for item_idx in history])
            else:
                one_sample['history'] = his_sep.join(history)
        data_samples.append(one_sample)
    return data_samples


In [ ]:
target_datasets = ["Beauty", "ML100K"] # 対象のデータセットを選択する [Beauty ML100K ML1M Yelp Electronics Movies CDs Clothing Taobao LastFM]

for dataset in target_datasets:
    for indexing in ["random", "sequential", "collaborative"]:
        generate_dataset(
            dataset = dataset,
            data_path = '/content/drive/MyDrive/OpenP5/data/',
            item_indexing = indexing,
            tasks_ = 'sequential,straightforward',
            prompt_file = 'OpenP5/prompt.txt',
        )
        generate_dataset_eval(
            dataset = dataset,
            data_path = '/content/drive/MyDrive/OpenP5/data/',
            item_indexing = indexing,
            tasks_ = 'sequential,straightforward',
            prompt_file = 'OpenP5/prompt.txt',
            mode="validation",
            prompt_str=str("seen:0")
        )
        generate_dataset_eval(
            dataset = dataset,
            data_path = '/content/drive/MyDrive/OpenP5/data/',
            item_indexing = indexing,
            tasks_ = 'sequential,straightforward',
            prompt_file = 'OpenP5/prompt.txt',
            mode="test",
            prompt_str=str("seen:0")
        )
        generate_dataset_eval(
            dataset = dataset,
            data_path = '/content/drive/MyDrive/OpenP5/data/',
            item_indexing = indexing,
            tasks_ = 'sequential,straightforward',
            prompt_file = 'OpenP5/prompt.txt',
            mode="test",
            prompt_str=str("unseen:0")
        )
